In [ ]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_curve, roc_auc_score
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from google.colab import drive

# 1. Mount Drive (Essential for accessing your saved data.zip)
drive.mount('/content/drive')

In [ ]:
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/data/cds" 


In [ ]:

file_path = f'{DATA_DIR}/bert_tokenized_tensors.pt'
print(f"Loading tensors from {file_path}...")

# weights_only=True fixes the PyTorch security warning
data = torch.load(file_path, weights_only=True) 

input_ids = data['input_ids']
attention_masks = data['attention_masks']
labels = data['labels']

print(f"Input IDs shape: {input_ids.shape}")
print(f"Attention Masks shape: {attention_masks.shape}")
print(f"Labels shape: {labels.shape}")



C:\Users\reese\AppData\Local\Temp\ipykernel_30056\4015265430.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load('../data/bert_tokenized_tensors.pt')


dict_keys(['input_ids', 'attention_masks', 'labels'])


torch.Size([562618, 256])
torch.Size([562618, 256])
torch.Size([562618])


In [ ]:
# 2. Define the Dataset Wrapper
class PostsDataset(Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_masks': self.attention_masks[idx],
            'labels': self.labels[idx]
        }
    
    def __len__(self):
        return len(self.labels)

# Instantiate the full dataset
full_dataset = PostsDataset(input_ids, attention_masks, labels)



In [ ]:
print("\nPerforming 70/15/15 dataset split...")

# Step A: Split into 70% Train and 30% Temporary (Validation + Test)
# We use stratify=labels.numpy() to ensure the 0/1 ratio remains perfectly balanced in all splits
train_idx, temp_idx = train_test_split(
    range(len(full_dataset)), 
    test_size=0.30, 
    stratify=labels.numpy(), 
    random_state=42
)

# Step B: Split the 30% Temporary subset evenly in half to get 15% Validation and 15% Test
val_idx, test_idx = train_test_split(
    temp_idx, 
    test_size=0.50, 
    stratify=[labels[i].item() for i in temp_idx], 
    random_state=42
)

# 4. Create PyTorch Subsets
train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)
test_dataset = Subset(full_dataset, test_idx)

print(f"Total dataset size: {len(full_dataset)}")
print(f"Train size: {len(train_dataset)} (70%)")
print(f"Val size:   {len(val_dataset)} (15%)")
print(f"Test size:  {len(test_dataset)} (15%)")

# 5. Create DataLoaders
# Batch size of 16 is standard for BERT, reduce to 8 if you run out of GPU memory
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("\nDataLoaders successfully created and ready for training!")

In [ ]:
# load BERT model

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

c:\Users\reese\Documents\SUTD stuff\work stuff\term 6\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\reese\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2321.55it/s]
BertForSequenceClassifi

In [14]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# freeze the pretrained layers
for param in model.bert.parameters():
    param.requires_grad = False

# keep only the classification head trainable
for param in model.classifier.parameters():
    param.requires_grad = True

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Trainable parameters: 1538


In [ ]:
# training function

from tqdm import tqdm

def train(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    total_preds = []
    total_labels = []

    for step, batch in enumerate(tqdm(dataloader, desc="Training")):
        # 1. MOVE TO GPU: Send the current batch of 32 rows to the GPU
        input_ids = batch["input_ids"].to(device)
        attention_masks = batch["attention_masks"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # 2. RUN ON GPU: The model processes the batch
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_masks,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits
        total_loss += loss.item() # .item() safely extracts the number from the GPU

        # 3. RUN ON GPU: Calculate gradients and update weights
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # 4. GET PREDICTIONS ON GPU: Find the highest probability class (0 or 1)
        preds = torch.argmax(logits, dim=1)

        # 5. MOVE BACK TO CPU: 
        # .detach()  -> Unhooks the data from the GPU's gradient graph
        # .cpu()     -> Moves it from GPU VRAM back to standard computer RAM
        # .numpy()   -> Converts it from a PyTorch tensor to a standard Python/NumPy list
        total_preds.extend(preds.detach().cpu().numpy())
        total_labels.extend(labels.detach().cpu().numpy())

    # 6. RUN ON CPU: Scikit-learn calculates the final score using the CPU arrays
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(total_labels, total_preds)
    f1score = f1_score(total_labels, total_preds)

    return avg_loss, accuracy, f1score



In [ ]:
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    total_preds = []
    total_labels = []

    with torch.no_grad(): # Disables gradient tracking entirely to save GPU memory
        for batch in tqdm(dataloader, desc="Evaluating"):
            # 1. MOVE TO GPU
            input_ids = batch["input_ids"].to(device)
            attention_masks = batch["attention_masks"].to(device)
            labels = batch["labels"].to(device)

            # 2. RUN ON GPU
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_masks,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits
            total_loss += loss.item()

            # 3. GET PREDICTIONS ON GPU
            preds = torch.argmax(logits, dim=1)

            # 4. MOVE BACK TO CPU for safe storage and metric calculation
            total_preds.extend(preds.detach().cpu().numpy())
            total_labels.extend(labels.detach().cpu().numpy())

    # 5. RUN ON CPU
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(total_labels, total_preds)
    f1score = f1_score(total_labels, total_preds)

    return avg_loss, accuracy, f1score

In [ ]:
import os

class EarlyStopping:
    def __init__(self, patience, mode='max', delta=0, save_dir="checkpoints", filename="best_model.pt"):
        self.patience = patience
        self.mode = mode 
        self.delta = delta
        
        # Create the directory if it doesn't exist
        self.save_dir = save_dir
        if not os.path.exists(self.save_dir):
            os.makedirs(self.save_dir)
            print(f"Created directory: {self.save_dir}")
            
        self.path = os.path.join(self.save_dir, filename)
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_score, model):
        if self.best_score is None:
            self.best_score = val_score
            self.save_checkpoint(model)
            return

        if self.mode == 'max':
            if val_score < self.best_score + self.delta:
                self.counter += 1
                print(f"EarlyStopping counter: {self.counter}/{self.patience}")
                if self.counter >= self.patience:
                    self.early_stop = True
            else:
                self.best_score = val_score
                self.save_checkpoint(model)
                self.counter = 0
                
        elif self.mode == 'min':
            if val_score > self.best_score - self.delta:
                self.counter += 1
                print(f"EarlyStopping counter: {self.counter}/{self.patience}")
                if self.counter >= self.patience:
                    self.early_stop = True
            else:
                self.best_score = val_score
                self.save_checkpoint(model)
                self.counter = 0

    def save_checkpoint(self, model):
        # Saves the model state_dict to the specified directory/file
        print(f"Validation improved. Saving best model to {self.path}...")
        torch.save(model.state_dict(), self.path)

In [ ]:
def training_loop(model, train_loader, val_loader, optimizer, early_stopping, num_epochs, device):
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    train_f1s, val_f1s = [], []

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        train_loss, train_acc, train_f1 = train(
            model, train_loader, optimizer, device
        )

        val_loss, val_acc, val_f1 = evaluate(
            model, val_loader, device
        )

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        train_f1s.append(train_f1)
        val_f1s.append(val_f1)

        early_stopping(val_acc, model)

        if early_stopping.early_stop:
            print("Early stopping triggered. Halting training.")
            break
            
    # Reload the best weights from the directory before finishing
    print(f"\nTraining Complete. Loading best weights from '{early_stopping.path}'...")
    model.load_state_dict(torch.load(early_stopping.path, weights_only=True))
    
    return train_losses, val_losses, train_accs, val_accs, train_f1s, val_f1s

In [ ]:
import os


CHECKPOINT_DIR = f"{DATA_DIR}/bert_checkpoints"
if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR)
    print(f"Created directory: {CHECKPOINT_DIR}")

# ==========================================
# PHASE 1: Train Only the Classification Head
# ==========================================
# Freeze all BERT parameters
for param in model.bert.parameters():
    param.requires_grad = False

# Keep only the final classifier layer trainable
for param in model.classifier.parameters():
    param.requires_grad = True

# Monitor validation accuracy (higher is better)
early_stopping = EarlyStopping(patience=3, mode='max', save_dir=CHECKPOINT_DIR, filename="phase1_head.pt")
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# Run Phase 1 (1 Epoch)
train_losses_1, val_losses_1, train_accs_1, val_accs_1, train_f1s_1, val_f1s_1 = training_loop(
    model, train_loader, val_loader, optimizer, early_stopping, 1, device
)

# ==========================================
# PHASE 2: Unfreeze Last 2 BERT Layers
# ==========================================
for param in model.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

# New EarlyStopping instance for this phase to track improvements properly
early_stopping = EarlyStopping(patience=3, mode='max', save_dir=CHECKPOINT_DIR, filename="phase2_unfrozen_2.pt")
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5) # Lower learning rate

# Run Phase 2 (3 Epochs)
train_losses_2, val_losses_2, train_accs_2, val_accs_2, train_f1s_2, val_f1s_2 = training_loop(
    model, train_loader, val_loader, optimizer, early_stopping, 3, device
)

# ==========================================
# PHASE 3: Unfreeze Last 4 BERT Layers
# ==========================================
for param in model.bert.encoder.layer[-4:-2].parameters():
    param.requires_grad = True
    
early_stopping = EarlyStopping(patience=3, mode='max', save_dir=CHECKPOINT_DIR, filename="phase3_unfrozen_4.pt")
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6) # Even lower learning rate

# Run Phase 3 (3 Epochs)
train_losses_3, val_losses_3, train_accs_3, val_accs_3, train_f1s_3, val_f1s_3 = training_loop(
    model, train_loader, val_loader, optimizer, early_stopping, 3, device
)


Epoch 1/1


100%|██████████| 352/352 [18:15<00:00,  3.11s/it]


Train Loss: 0.5320
Train Acc:  0.7749
Train F1 score:    0.7728
Val Loss:   0.5035
Val Acc:    0.7911
Val F1 score:    0.8047
Validation improved. Saving model...

Epoch 1/3


100%|██████████| 352/352 [18:06<00:00,  3.09s/it]


Train Loss: 0.1222
Train Acc:  0.9554
Train F1 score:    0.9541
Val Loss:   0.0687
Val Acc:    0.9774
Val F1 score:    0.9767
Validation improved. Saving model...

Epoch 2/3


100%|██████████| 352/352 [17:25<00:00,  2.97s/it]


Train Loss: 0.0648
Train Acc:  0.9784
Train F1 score:    0.9778
Val Loss:   0.0747
Val Acc:    0.9794
Val F1 score:    0.9790
Validation improved. Saving model...

Epoch 3/3


 44%|████▍     | 469/1055 [39:57<50:55,  5.21s/it]  

In [ ]:
# plot graphs

iters = list(range(1, 8))

total_train_losses = train_losses_1 + train_losses_2 + train_losses_3
total_train_accs = train_accs_1 + train_accs_2 + train_accs_3
total_train_f1s = train_f1s_1 + train_f1s_2 + train_f1s_3
total_val_losses = val_losses_1 + val_losses_2 + val_losses_3
total_val_accs = val_accs_1 + val_accs_2 + val_accs_3
total_val_f1s = val_f1s_1 + val_f1s_2 + val_f1s_3

fig, ax = plt.subplots(1, 3)
ax[0,0].plot(iters, total_train_losses, marker='o', label='Training Loss')
ax[0,0].plot(iters, total_val_losses, marker='s', label='Validation Loss')
ax[0,1].plot(iters, total_train_accs, marker='o', label='Training Accuracy')
ax[0,1].plot(iters, total_val_accs, marker='s', label='Validation Accuracy')
ax[0,2].plot(iters, total_train_f1s, marker='o', label='Training F1 Score')
ax[0,2].plot(iters, total_val_f1s, marker='s', label='Validation F1 Score')

plt.legend()
plt.show()


<class 'list'>


In [ ]:
def evaluate_on_testing_set(y_test, y_pred):
  # Calculate AUC
  print("AUC is: ", roc_auc_score(y_test, y_pred))

  # print out recall and precision
  print(classification_report(y_test, y_pred))

  # print out confusion matrix
  print("Confusion Matrix: \n", confusion_matrix(y_test, y_pred))

  # # calculate points for ROC curve
  fpr, tpr, thresholds = roc_curve(y_test, y_pred)

  # Plot ROC curve
  plt.plot(fpr, tpr, label='ROC curve (area = %0.3f)' % roc_auc_score(y_test, y_pred))
  plt.plot([0, 1], [0, 1], 'k--')  # random predictions curve
  plt.xlim([0.0, 1.0])
  plt.ylim([0.0, 1.0])
  plt.xlabel('False Positive Rate or (1 - Specifity)')
  plt.ylabel('True Positive Rate or (Sensitivity)')
  plt.title('Receiver Operating Characteristic')

In [ ]:
# testing

model.eval()

test_loss = 0
test_preds = []
test_labels = []

with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_masks = batch["attention_masks"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_masks,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        test_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        test_preds.extend(preds.detach().cpu().numpy())
        test_labels.extend(labels.detach().cpu().numpy())

In [ ]:
# testing metrics

print("Test Loss:", test_loss)
evaluate_on_testing_set(test_labels, test_preds)